# McMiner Benchmark Exploration

This notebook explores the officially retrieved McMiner benchmark to establish its schema, size, distribution, and suitability for Re:Learn.

In [ ]:
import os
import json
import pandas as pd
import glob

DATA_DIR = '../data/raw/mcminer'

print("Contents of dataset directory:", os.listdir(DATA_DIR))

## 1. Inspect Misconception Bank

In [ ]:
with open(os.path.join(DATA_DIR, 'misconception_bank.json'), 'r') as f:
    misc_bank = json.load(f)

print(f"Number of misconception categories defined: {len(misc_bank)}")
example_key = list(misc_bank.keys())[0]
print(f"Example key: {example_key}")
print("Fields:", misc_bank[example_key])

## 2. Inspect Problems

In [ ]:
with open(os.path.join(DATA_DIR, 'problems_processed.json'), 'r') as f:
    problems = json.load(f)

print(f"Number of problems: {len(problems)}")
example_prob = list(problems.keys())[0]
print(f"Example problem keys: {list(problems[example_prob].keys())}")

## 3. Extracting Code Examples

The source codes exhibiting misconceptions are stored in `corrupted_codes_best/`.

In [ ]:
code_files = glob.glob(os.path.join(DATA_DIR, 'corrupted_codes_best', '*.json'))
code_files = [f for f in code_files if not f.endswith('filtering_report.json')]
print(f"Found {len(code_files)} code example files.")

all_examples = []
for cf in code_files:
    with open(cf, 'r') as f:
        data = json.load(f)
        for example in data:
            all_examples.append(example)

df = pd.DataFrame(all_examples)
print(f"Total Code Examples: {len(df)}")
print("Dataset Schema:", df.columns.tolist())


## 4. Class Distribution and Missing Values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nUnique Programming Languages:")
if 'language' in df.columns:
    print(df['language'].value_counts())
else:
    print("Language field not explicit; assume Python based on paper/repo.")

print("\nMisconception Category Distribution:")
if 'misc_id' in df.columns:
    print(df['misc_id'].value_counts().describe())
    print(f"Number of unique misconception labels: {df['misc_id'].nunique()}")


## 5. Duplicate Detection and Splits

In [ ]:
if 'corrupted_code' in df.columns:
    print(f"Number of exact duplicate code strings: {df['corrupted_code'].duplicated().sum()}")
else:
    print("corrupted_code column not found.")

print("Check for existing train/test split field:")
split_cols = [c for c in df.columns if 'split' in c.lower() or 'train' in c.lower()]
if split_cols:
    print("Found split columns:", split_cols)
else:
    print("No explicit train/test split found in the dataset schema.")